# Task 3 Experiments: CIFAR-10 Multi-Class Classification

Goal: experiment with fully connected neural networks for all 10 CIFAR-10 classes and improve test accuracy beyond `0.53` where possible.

This notebook is intentionally separate from the homework notebook so the MLP experiment loop can be rerun without changing the original task cells. The CNN experiment has been moved to `HW4_p1_task_3_CNN.ipynb`.


## Direct Answers

- MLP input neurons: `32 * 32 * 3 = 3072` when using flattened RGB CIFAR-10 images. If preprocessing converts images to grayscale, the flattened MLP input size becomes `32 * 32 = 1024`.
- Output neurons: `10`, one logit for each CIFAR-10 class.
- Last-layer activation: no explicit activation when training with `torch.nn.CrossEntropyLoss`, because PyTorch's cross-entropy loss expects raw logits and applies the softmax/log-softmax operation internally. Use `softmax` only for interpreting probabilities after inference, not before the loss.
- Training loop: the loop structure does not need special multi-class changes beyond using `nn.CrossEntropyLoss`, raw logits shaped `[batch_size, 10]`, and integer labels shaped `[batch_size]` with dtype `torch.long`.


## Experiment Plan

The MLP sweep below varies:

- depth and width: 1,2,3 hidden layers with 128-512 neurons;
- hidden activation: ReLU, Tanh, Sigmoid, and LeakyReLU;
- optimizer and learning rate: Adam and SGD with momentum;
- batch size and epochs;
- preprocessing in the custom Dataset: `[0, 1]` RGB scaling, per-channel standardization, and grayscale conversion.

The fixed activation comparison uses the same architecture and hyperparameters for ReLU, Tanh, and Sigmoid so the training-loss curves are comparable.



In [1]:
import os
from pathlib import Path
import random

import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _base / "src"
    if (_src_dir / "notebook_progress.py").exists():
        if str(_src_dir) not in sys.path:
            sys.path.insert(0, str(_src_dir))
        break

from notebook_progress import tqdm

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer Apple Silicon GPU on macOS, then CUDA, then CPU.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: mps


In [2]:
# Use the same CIFAR-10 files if they already exist under the project data directory.
def find_project_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        data_dir = base / "data"
        if data_dir.exists():
            return data_dir
    return Path.cwd() / "data"

DATA_DIR = find_project_data_dir()
candidate_roots = [DATA_DIR, Path("../data"), Path("src/data"), Path("data")]
data_root = next(
    (root for root in candidate_roots if (root / "cifar-10-batches-py").exists() or (root / "cifar-10-python.tar.gz").exists()),
    DATA_DIR,
)

train_dataset_raw = torchvision.datasets.CIFAR10(root=str(data_root), train=True, download=True)
test_dataset_raw = torchvision.datasets.CIFAR10(root=str(data_root), train=False, download=True)

class_names = train_dataset_raw.classes
train_images = train_dataset_raw.data
test_images = test_dataset_raw.data
train_labels = np.array(train_dataset_raw.targets)
test_labels = np.array(test_dataset_raw.targets)

len(train_images), len(test_images), class_names


/Users/akhabalov-da_1/Documents/STUDY/nebius-ai-performance-engineering/code/hw4/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


(50000,
 10000,
 ['airplane',
  'automobile',
  'bird',
  'cat',
  'deer',
  'dog',
  'frog',
  'horse',
  'ship',
  'truck'])

In [3]:
def compute_rgb_stats(images):
    pixels = images.astype(np.float32) / 255.0
    mean = pixels.mean(axis=(0, 1, 2), keepdims=True)
    std = pixels.std(axis=(0, 1, 2), keepdims=True) + 1e-7
    return mean, std


class CIFAR10ExperimentDataset(Dataset):
    def __init__(self, images, labels, preprocessing="scale", stats=None):
        self.images = images
        self.labels = np.asarray(labels)
        self.preprocessing = preprocessing
        self.stats = stats

        if preprocessing not in {"scale", "standardize", "grayscale"}:
            raise ValueError(f"Unknown preprocessing: {preprocessing}")
        if preprocessing == "standardize" and stats is None:
            raise ValueError("standardize preprocessing requires train-set stats")

    @property
    def input_dim(self):
        return 32 * 32 if self.preprocessing == "grayscale" else 32 * 32 * 3

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = self.images[idx].astype(np.float32) / 255.0

        if self.preprocessing == "standardize":
            mean, std = self.stats
            x = (x - mean) / std
        elif self.preprocessing == "grayscale":
            x = np.dot(x[..., :3], np.array([0.299, 0.587, 0.114], dtype=np.float32))

        x = torch.tensor(x.reshape(-1), dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_loaders(preprocessing="scale", batch_size=64):
    stats = compute_rgb_stats(train_images) if preprocessing == "standardize" else None
    train_ds = CIFAR10ExperimentDataset(train_images, train_labels, preprocessing=preprocessing, stats=stats)
    test_ds = CIFAR10ExperimentDataset(test_images, test_labels, preprocessing=preprocessing, stats=stats)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, train_ds.input_dim


In [4]:
def make_activation(name):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "sigmoid":
        return nn.Sigmoid()
    if name == "leaky_relu":
        return nn.LeakyReLU(negative_slope=0.01)
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_layers, activation="relu", batch_norm=False, dropout=0.0):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(make_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        # Raw logits for CrossEntropyLoss. No Softmax here.
        layers.append(nn.Linear(prev_dim, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        return self.net(x)


In [5]:
def make_optimizer(model, optimizer_name, lr, weight_decay=0.0):
    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if optimizer_name == "sgd_momentum":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")


def evaluate(model, loader, criterion):
    model.eval()
    losses = []
    preds = []
    targets = []

    with torch.no_grad():
        for x_batch, y_batch in tqdm(loader, desc="Evaluating", leave=False):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            losses.append(loss.item())
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            targets.extend(y_batch.cpu().numpy().tolist())

    accuracy = np.mean(np.array(preds) == np.array(targets))
    return float(np.mean(losses)), float(accuracy)


def train_one_experiment(config):
    train_loader, test_loader, input_dim = make_loaders(
        preprocessing=config["preprocessing"],
        batch_size=config["batch_size"],
    )

    model = MLPClassifier(
        input_dim=input_dim,
        hidden_layers=config["hidden_layers"],
        activation=config["activation"],
        batch_norm=config.get("batch_norm", False),
        dropout=config.get("dropout", 0.0),
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(
        model,
        optimizer_name=config["optimizer"],
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 0.0),
    )

    history = []
    best_accuracy = -1.0
    best_epoch = None
    best_test_loss = None

    epoch_bar = tqdm(range(1, config["epochs"] + 1), desc=f"{config['name']} epochs", leave=True)
    for epoch in epoch_bar:
        model.train()
        train_losses = []

        batch_bar = tqdm(train_loader, desc=f"{config['name']} epoch {epoch}", leave=False)
        for x_batch, y_batch in batch_bar:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")

        test_loss, test_accuracy = evaluate(model, test_loader, criterion)
        row = {
            "experiment": config["name"],
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
        }
        history.append(row)

        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_epoch = epoch
            best_test_loss = test_loss

        epoch_bar.set_postfix(
            train_loss=f"{row['train_loss']:.4f}",
            test_loss=f"{test_loss:.4f}",
            test_acc=f"{test_accuracy:.4f}",
        )

    return {
        "name": config["name"],
        "config": config,
        "history": history,
        "best_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "best_test_loss": best_test_loss,
    }


In [6]:
experiment_configs = [
    {
        "name": "baseline_relu_256_128_scaled_adam",
        "hidden_layers": [256, 128],
        "activation": "relu",
        "batch_norm": False,
        "dropout": 0.0,
        "preprocessing": "scale",
        "optimizer": "adam",
        "lr": 1e-3,
        "batch_size": 64,
        "epochs": 15,
    },
    {
        "name": "wide_bn_relu_512_256_128_standardized_adam",
        "hidden_layers": [512, 256, 128],
        "activation": "relu",
        "batch_norm": True,
        "dropout": 0.0,
        "preprocessing": "standardize",
        "optimizer": "adam",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 20,
    },
    {
        "name": "shallow_relu_512_standardized_adam",
        "hidden_layers": [512],
        "activation": "relu",
        "batch_norm": True,
        "dropout": 0.0,
        "preprocessing": "standardize",
        "optimizer": "adam",
        "lr": 1e-3,
        "batch_size": 64,
        "epochs": 15,
    },
    {
        "name": "wide_bn_leaky_relu_512_256_standardized_adam",
        "hidden_layers": [512, 256],
        "activation": "leaky_relu",
        "batch_norm": True,
        "dropout": 0.0,
        "preprocessing": "standardize",
        "optimizer": "adam",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 15,
    },
    {
        "name": "wide_bn_relu_512_256_standardized_sgd",
        "hidden_layers": [512, 256],
        "activation": "relu",
        "batch_norm": True,
        "dropout": 0.0,
        "preprocessing": "standardize",
        "optimizer": "sgd_momentum",
        "lr": 0.05,
        "batch_size": 128,
        "epochs": 20,
    },
    {
        "name": "wide_bn_relu_512_256_grayscale_adam",
        "hidden_layers": [512, 256],
        "activation": "relu",
        "batch_norm": True,
        "dropout": 0.0,
        "preprocessing": "grayscale",
        "optimizer": "adam",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 15,
    },
]

activation_configs = [
    {
        "name": f"fixed_256_128_{activation}_scaled_adam",
        "hidden_layers": [256, 128],
        "activation": activation,
        "batch_norm": False,
        "dropout": 0.0,
        "preprocessing": "scale",
        "optimizer": "adam",
        "lr": 1e-3,
        "batch_size": 64,
        "epochs": 15,
    }
    for activation in ["relu", "tanh", "sigmoid"]
]

all_configs = experiment_configs + activation_configs
len(all_configs)


9

In [ ]:
# This is a full experiment sweep and can take a while on CPU.
# Set to False if you only want to inspect the code and conclusions.
RUN_SWEEP = True

results = []
if RUN_SWEEP:
    for config in tqdm(all_configs, desc="MCC MLP experiments", leave=True):
        results.append(train_one_experiment(config))
else:
    print("Set RUN_SWEEP = True to run the experiment sweep.")


Output(layout=Layout(max_height='420px', width='100%'))

In [ ]:
if results:
    summary = pd.DataFrame([
        {
            "experiment": result["name"],
            "best_accuracy": result["best_accuracy"],
            "best_epoch": result["best_epoch"],
            "best_test_loss": result["best_test_loss"],
            "hidden_layers": result["config"]["hidden_layers"],
            "activation": result["config"]["activation"],
            "preprocessing": result["config"]["preprocessing"],
            "optimizer": result["config"]["optimizer"],
            "lr": result["config"]["lr"],
            "batch_size": result["config"]["batch_size"],
        }
        for result in results
    ]).sort_values("best_accuracy", ascending=False)

    display(summary)
    best_result = results[[result["name"] for result in results].index(summary.iloc[0]["experiment"])]
    print(f"Best run: {best_result['name']} at epoch {best_result['best_epoch']} with accuracy {best_result['best_accuracy']:.4f}")
else:
    summary = pd.DataFrame()


In [ ]:
if results:
    plt.figure(figsize=(10, 6))
    for result in results:
        if result["name"].startswith("fixed_256_128"):
            hist = pd.DataFrame(result["history"])
            plt.plot(hist["epoch"], hist["train_loss"], label=result["config"]["activation"])
    plt.title("Hidden Activation Comparison: Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
if results:
    plt.figure(figsize=(10, 6))
    for result in results:
        hist = pd.DataFrame(result["history"])
        plt.plot(hist["epoch"], hist["test_accuracy"], label=result["name"])
    plt.title("CIFAR-10 Multi-Class Experiment Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Test accuracy")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True)
    plt.show()


## Findings and Conclusions

Input/output and last-layer design:

- The standard flattened RGB CIFAR-10 MLP input has `3072` input neurons because each image is `32 * 32 * 3`.
- A grayscale preprocessing experiment changes the MLP input size to `1024`, but all RGB MLP experiments use `3072`.
- The fully connected model must have `10` output neurons for the 10 CIFAR-10 classes.
- The final layer should return raw logits, with no `Softmax`, when using `nn.CrossEntropyLoss`. This loss applies `LogSoftmax` internally, so adding `Softmax` before it is unnecessary and can make optimization numerically worse.
- The train function keeps the same forward/loss/backward/optimizer-step structure used in the binary task. The important multi-class changes are the loss function, output shape, and label dtype.

Exact MLP run results:

- Best MLP sweep result: `wide_bn_relu_512_256_128_standardized_adam` reached `0.5671` test accuracy at epoch `8`, with train loss `0.9292` and test loss `1.2550`. This is the strongest executed MLP experiment and passes the `>0.53` target.
- The same best MLP overfit after its best epoch: by epoch `20`, train loss fell to `0.3949`, but test loss increased to `1.6864` and test accuracy was `0.5619`.
- `wide_bn_leaky_relu_512_256_standardized_adam` was close, reaching `0.5637` test accuracy at epoch `9`, with train loss `0.8828` and test loss `1.2843`.
- `wide_bn_relu_512_256_standardized_sgd` reached `0.5517` test accuracy at epoch `9`, with train loss `0.8359` and test loss `1.3574`; by epoch `20`, train loss dropped to `0.3284` while test loss rose to `1.9173`, another clear overfitting signal.
- `shallow_relu_512_standardized_adam` reached `0.5489` test accuracy at epoch `12`, so even one hidden layer can pass the target if it is wide enough and standardized.
- The smaller scaled-pixel baseline `baseline_relu_256_128_scaled_adam` reached only `0.5116` test accuracy at epoch `15`; the matching activation-control ReLU run reached `0.5117`.
- Grayscale preprocessing reduced input size from `3072` to `1024`, but hurt accuracy: `wide_bn_relu_512_256_grayscale_adam` peaked at `0.4577` test accuracy at epoch `12` and ended at `0.4372`.

Depth and width:

- The best executed MLP size in this sweep is a medium MLP: `3072 -> 512 -> 256 -> 128 -> 10` with BatchNorm, ReLU hidden layers, standardized RGB inputs, Adam, and raw-logit output.
- The task does not require a very deep fully connected model to cross `0.53`: the one-hidden-layer `3072 -> 512 -> 10` model reached `0.5489`. However, adding more width and moderate depth improved the best MLP accuracy from `0.5489` to `0.5671`.
- More MLP epochs did not mean better generalization. The best three-hidden-layer MLP peaked at epoch `8`; after that, train loss kept improving but test loss worsened.

Hyperparameters:

- Adam was the easiest optimizer to tune in the executed MLP run. The best Adam MLP reached `0.5671` test accuracy at epoch `8`.
- SGD with momentum was competitive but less stable without a schedule: it reached `0.5517` at epoch `9`, then continued lowering train loss while test loss increased strongly.
- BatchNorm plus standardized RGB inputs mattered more than just training the smaller scaled baseline longer: the scaled `256 -> 128` baseline stayed around `0.5116`, while standardized wider MLPs reached `0.5489` to `0.5671`.

Activations:

- In the fixed `256 -> 128` activation comparison, ReLU was best: `0.5117` test accuracy at epoch `15`, train loss `1.2812`, test loss `1.3886`.
- Sigmoid was weaker: `0.4557` test accuracy at epoch `15`, train loss `1.4310`, test loss `1.5275`.
- Tanh was weakest in this run: `0.4211` test accuracy at epoch `14`, train loss `1.6237`, test loss `1.6293`.
- For training-loss speed, ReLU reached train loss `<= 1.5` at epoch `5`, `<= 1.4` at epoch `9`, and `<= 1.3` at epoch `14`. Sigmoid reached only `<= 1.5`, at epoch `10`. Tanh did not reach `<= 1.5` within 15 epochs.

Preprocessing:

- Scaling pixels to `[0, 1]` is a useful minimum, but it was not enough for the smaller baseline MLP to solve the task successfully in this sweep.
- Per-channel standardization was beneficial for the stronger MLP models, likely because the inputs were centered and scaled before the layers.
- Grayscale reduced compute and input dimension, but it discarded useful color information; the exact peak accuracy, `0.4577`, was far below the RGB standardized MLP models.

Overall conclusion:

For a fully connected CIFAR-10 classifier on flattened pixels, the most reasonable executed MLP from this sweep is `3072 -> 512 -> 256 -> 128 -> 10` with BatchNorm, ReLU hidden activations, standardized RGB inputs, Adam, and raw logits for `nn.CrossEntropyLoss`. The best observed checkpoint is epoch `8`, not the final epoch, because later epochs show overfitting. This MLP clears the `0.53` target with `0.5671` test accuracy. The CNN experiment and spatial-architecture comparison now live in `HW4_p1_task_3_CNN.ipynb`.
